In [3]:
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from torch.utils.data import DataLoader
from torchvision import models

from vasijas.config import load_config
from vasijas.evaluacion import DatasetTestConRuta
from vasijas.paths import ProjectPaths
from vasijas.metadata import cargar_metadata
from vasijas.entrenamiento_cnn import (
    ConfiguracionTransformacionesImagen,
    crear_datasets_imagen_desde_metadata,
    crear_transformaciones_imagen,
    seleccionar_dispositivo_procesamiento,
)

In [4]:
CONFIG = load_config()
PATHS = ProjectPaths.from_root(CONFIG.project_root)
DATA_DIR = PATHS.DATA_UNIFICADO_DIVIDIDO
MODEL_PATH = (
    PATHS.OUTPUT_MODELS
    / "model_resnet18_unificado_bs16_lr1e-3_weighted_unfreeze.pth"
)
CSV_PATH = PATHS.OUTPUT_PREDICTIONS / f"predicciones_test_{MODEL_PATH.stem}.csv"
BATCH_SIZE = 16
DEVICE = seleccionar_dispositivo_procesamiento()
print(f"Dispositivo: {DEVICE}")

Dispositivo: cuda


In [5]:
# Se reutiliza el split persistido: no se vuelve a dividir el dataset.
metadata_split = cargar_metadata(DATA_DIR)
if "Identificador" not in metadata_split or metadata_split["Identificador"].isna().any():
    raise ValueError("La metadata debe incluir un Identificador para cada imagen.")

# Resize a 224 x 224 y normalización ImageNet, sin aumentación en test.
transformaciones = crear_transformaciones_imagen(
    ConfiguracionTransformacionesImagen(tamano_imagen=224)
)
datasets_imagen = crear_datasets_imagen_desde_metadata(
    data_dir=DATA_DIR,
    metadata_split=metadata_split,
    transformacion_entrenamiento=transformaciones[0],
    transformacion_validacion=transformaciones[1],
    transformacion_test=transformaciones[2],
)
nombres_clases = datasets_imagen.class_names
metadata_por_ruta = metadata_split.assign(
    ruta_relativa=metadata_split["ruta_relativa"].map(lambda ruta: Path(ruta).as_posix())
).set_index("ruta_relativa")

In [6]:

test_loader = DataLoader(
    DatasetTestConRuta(datasets_imagen.test, DATA_DIR),
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=0,
    pin_memory=DEVICE.type == "cuda",
)
cantidad_test = int(metadata_split["split"].eq("test").sum())
assert len(test_loader.dataset) == cantidad_test
print(f"Imágenes de prueba: {cantidad_test}")
print("Clases (orden de ImageFolder usado en entrenamiento):", nombres_clases)

# Metadata original: sus columnas se conservan sin sustituir valores faltantes.
metadata_unificado = cargar_metadata(PATHS.DATA_UNIFICADO)
if "Identificador" not in metadata_unificado:
    raise ValueError("La metadata original no incluye la columna Identificador.")
if metadata_unificado["Identificador"].isna().any():
    raise ValueError("La metadata original contiene identificadores nulos.")
if metadata_unificado["Identificador"].duplicated().any():
    raise ValueError("La metadata original contiene identificadores duplicados.")

identificadores_test = metadata_split.loc[
    metadata_split["split"].eq("test"), "Identificador"
]
if identificadores_test.duplicated().any():
    raise ValueError("Hay identificadores de prueba duplicados en el split.")
faltantes = set(identificadores_test) - set(metadata_unificado["Identificador"])
if faltantes:
    raise ValueError(f"Falta metadata original para estas imágenes: {sorted(faltantes)[:10]}")

Imágenes de prueba: 3059
Clases (orden de ImageFolder usado en entrenamiento): ['clase01', 'clase02', 'clase03', 'clase05', 'clase06', 'clase07', 'clase08', 'clase09', 'clase10', 'clase11']


In [7]:
# El archivo contiene un state_dict: se reconstruye la arquitectura sin descargar pesos.
state_dict = torch.load(MODEL_PATH, map_location="cpu", weights_only=True)
num_classes = state_dict["fc.weight"].shape[0]
if num_classes != len(nombres_clases):
    raise ValueError(
        f"El modelo tiene {num_classes} salidas y el dataset {len(nombres_clases)} clases."
    )

model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model.load_state_dict(state_dict, strict=True)
model = model.to(DEVICE)
model.eval()
print(f"Modelo cargado: {MODEL_PATH.name}")


Modelo cargado: model_resnet18_unificado_bs16_lr1e-3_weighted_unfreeze.pth


In [8]:
filas = []

model.eval()
with torch.inference_mode():
    for numero_lote, (imagenes, etiquetas, rutas) in enumerate(test_loader, start=1):
        salidas = model(imagenes.to(DEVICE, non_blocking=DEVICE.type == "cuda"))
        predicciones = salidas.argmax(dim=1).cpu().tolist()

        for ruta, real, predicha in zip(
            rutas, etiquetas.tolist(), predicciones, strict=True
        ):
            filas.append(
                {
                    "Identificador": metadata_por_ruta.at[ruta, "Identificador"],
                    "ruta_relativa": ruta,
                    "clase_real": nombres_clases[real],
                    "clase_predicha": nombres_clases[predicha],
                    "acierto": real == predicha,
                }
            )
        if numero_lote % 25 == 0 or numero_lote == len(test_loader):
            print(f"Procesadas {len(filas)}/{cantidad_test} imágenes")

predicciones_test = pd.DataFrame(
    filas,
    columns=["Identificador", "ruta_relativa", "clase_real", "clase_predicha", "acierto"],
)

# Verificar que cada imagen de test tenga exactamente una predicción.
rutas_test = set(metadata_split.loc[metadata_split["split"].eq("test"), "ruta_relativa"])
assert len(predicciones_test) == cantidad_test
assert predicciones_test["ruta_relativa"].is_unique
assert set(predicciones_test["ruta_relativa"]) == rutas_test
assert not predicciones_test.isna().any().any()
assert predicciones_test["acierto"].equals(predicciones_test["clase_real"].eq(predicciones_test["clase_predicha"]))

Procesadas 400/3059 imágenes
Procesadas 800/3059 imágenes
Procesadas 1200/3059 imágenes
Procesadas 1600/3059 imágenes
Procesadas 2000/3059 imágenes
Procesadas 2400/3059 imágenes
Procesadas 2800/3059 imágenes
Procesadas 3059/3059 imágenes


In [9]:
# Unir por identificador, nunca por posición: el orden de los CSV puede diferir.
tabla = predicciones_test[["Identificador", "clase_predicha", "acierto"]].merge(
    metadata_unificado,
    on="Identificador",
    how="left",
    sort=False,
    validate="one_to_one",
    indicator=True,
)
assert tabla["_merge"].eq("both").all()
tabla = tabla[list(metadata_unificado.columns) + ["clase_predicha", "acierto"]]
assert len(tabla) == cantidad_test
assert tabla["Identificador"].is_unique

# Comprobar que todas las columnas originales mantienen los valores de la fuente.
pd.testing.assert_frame_equal(
    tabla[list(metadata_unificado.columns)].reset_index(drop=True),
    metadata_unificado.set_index("Identificador", drop=False)
    .loc[tabla["Identificador"]]
    .reset_index(drop=True),
)

CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
tabla.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
print(f"Filas: {len(tabla)} | Columnas: {len(tabla.columns)}")
print(f"Aciertos: {int(tabla['acierto'].sum())}/{len(tabla)} ({tabla['acierto'].mean():.2%})")
print(f"Tabla completa guardada en: {CSV_PATH}")
display(tabla)


Filas: 3059 | Columnas: 18
Aciertos: 2377/3059 (77.71%)
Tabla completa guardada en: /home/rafael/Dev/vasijas/output/predictions/predicciones_test_model_resnet18_unificado_bs16_lr1e-3_weighted_unfreeze.csv


,Identificador,Tiene asa,Tendencia Formal,Borde,Cuello,Cuerpo,Base,Clase Ana,Diametro Borde,Diametro Base,Altura Total,Cronologia,Ubicacion,id_perfil_completo,borde_origen,porcentaje_avance,clase_predicha,acierto
0,ba_191_4,Si,IV,7,B,4,4,1.0,128,38,409,ibero,Baza,ba_191_4,completo,100.0,clase01,True
1,ba_191_4_base_10_centrado,Si,IV,7,B,4,4,1.0,128,38,409,ibero,Baza,ba_191_4,base,10.0,clase01,True
2,ba_191_4_base_20_centrado,Si,IV,7,B,4,4,1.0,128,38,409,ibero,Baza,ba_191_4,base,20.0,clase01,True
3,ba_191_4_base_30_centrado,Si,IV,7,B,4,4,1.0,128,38,409,ibero,Baza,ba_191_4,base,30.0,clase01,True
4,ba_191_4_base_40_centrado,Si,IV,7,B,4,4,1.0,128,38,409,ibero,Baza,ba_191_4,base,40.0,clase01,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3054,tu_91_3_tope_50_centrado,No,II,1,B,2,1,11.0,181,85,44,ibero,Tútugi,tu_91_3,tope,50.0,clase11,True
3055,tu_91_3_tope_60_centrado,No,II,1,B,2,1,11.0,181,85,44,ibero,Tútugi,tu_91_3,tope,60.0,clase11,True
3056,tu_91_3_tope_70_centrado,No,II,1,B,2,1,11.0,181,85,44,ibero,Tútugi,tu_91_3,tope,70.0,clase11,True
3057,tu_91_3_tope_80_centrado,No,II,1,B,2,1,11.0,181,85,44,ibero,Tútugi,tu_91_3,tope,80.0,clase11,True


La tabla `tabla` contiene una fila por imagen de prueba con todas las columnas originales de `data/11.unificado/metadata.csv`, en el mismo orden, seguidas de `clase_predicha` y `acierto` (`True` si acertó, `False` si no). La unión se realiza mediante `Identificador`, valida una única coincidencia por imagen y conserva los valores faltantes de la metadata original. El CSV completo se guarda en `output/predictions/`. Para regenerarlo, ejecutar las celdas desde el principio.